# Recorda entre celdas: simulación de uso

Laboratorio de `uibcdf/recorda-lab#3`, enlazado con `uibcdf/recorda#1`.
La librería dummy conserva sus resultados y no importa Recorda. Solo los consumidores
de `experiments/consumer.py` declaran límites semánticos con decoradores.

Ejecuta las celdas en orden con el entorno de desarrollo Python 3.14 y ambos paquetes
instalados. Cada ejecución completa crea un directorio nuevo. Los registros cubren
solo las operaciones declaradas; activar una sesión no observa llamadas arbitrarias.

In [ ]:
import json
import sys
from pathlib import Path
from uuid import uuid4

import recorda

lab_root = Path.cwd()
if lab_root.name == "notebooks":
    lab_root = lab_root.parent
assert (lab_root / "experiments").is_dir(), "Abre el notebook desde recorda-lab"
sys.path.insert(0, str(lab_root / "experiments"))
run_dir = lab_root / "artifacts" / f"notebook-{uuid4().hex}"
run_dir.mkdir(parents=True, exist_ok=False)
print(sys.executable)
print(run_dir)

In [ ]:
from consumer import REFERENCE_ADAPTERS, analyze, sample_count, summarize_samples

from recorda_lab import summarize

samples = [1, 2, 3, 4]
expected = summarize(samples)
assert summarize_samples(samples) == expected
assert not list(run_dir.glob("*.jsonl"))
(run_dir / "samples.json").write_text(json.dumps(samples))

Activa una vez. Las siguientes celdas llaman a las funciones normalmente.

In [ ]:
session = recorda.start(
    "notebook_analysis",
    path=run_dir / "analysis.jsonl",
    reference_adapters=REFERENCE_ADAPTERS,
    gaps=["dummy internals and native file persistence are unobserved"],
)

In [ ]:
result = analyze(samples)
assert result == expected
print(result.to_dict())
(run_dir / "native-result.json").write_text(json.dumps(result.to_dict(), sort_keys=True))

In [ ]:
assert sample_count(samples) == 4
ongoing = recorda.inspect(run_dir / "analysis.jsonl")
assert ongoing.status == "incomplete"  # Todavía no hemos cerrado la sesión.
assert len(ongoing.operations) == 3
print([(operation["name"], operation["status"]) for operation in ongoing.operations])

Cierra en otra celda. El resultado sigue perteneciendo a la librería dummy.

In [ ]:
record = recorda.stop()
assert record.status == "succeeded"
parent, child, count = record.operations
assert child["parent_id"] == parent["id"]
assert child["outputs"]["return"]["identifier"] == result.identifier
assert summarize_samples(samples) == expected
assert len(recorda.inspect(run_dir / "analysis.jsonl").operations) == 3
print(record.status)

## Un fallo también pertenece al registro

Capturamos aquí la excepción conocida para que se pueda ejecutar todo el notebook.
El runner automatizado prueba también una excepción sin capturar entre celdas.
Cerrar después no transforma un fallo en éxito.

In [ ]:
recorda.start(
    "failed_analysis", path=run_dir / "failure.jsonl", reference_adapters=REFERENCE_ADAPTERS
)

In [ ]:
try:
    summarize_samples([])
except ValueError as error:
    print(type(error).__name__)
else:
    raise AssertionError("La operación dummy debía fallar")

In [ ]:
failed_record = recorda.stop()
assert failed_record.status == "failed"
assert failed_record.operations[0]["exception"]["type"] == "builtins.ValueError"
print(failed_record.status)

## Interrupción y pérdida del kernel

`python experiments/run_notebook.py artifacts/kernel-check` ejecuta un kernel desechable:
comprueba `await`, rechazo del cierre por una tarea hija, SIGINT durante una operación y
pérdida del kernel sin `stop()`. Conserva `executed.ipynb`, los journals y `acceptance.json`.
El journal sin cerrar permanece `incomplete`; no se fabrica un final exitoso.
Estas pruebas no cierran el kernel en el que estás leyendo este notebook.

Los resultados califican la combinación concreta de IPykernel/cliente probada. La
ejecución automatizada no prueba por sí sola todas las interfaces de navegador.